# AutoResearch-MRL: Results Analysis

This notebook reads `results.tsv` and generates analysis plots and tables.

Run this after experiments to visualize progress and compare policies.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Load results
df = pd.read_csv('results.tsv', sep='\t')
print(f'Total experiments: {len(df)}')
print(f'\nStatus counts:')
print(df['status'].value_counts())
df.head(10)

## 1. Baseline Comparison

In [ ]:
# Baseline results
baselines = df[df['status'] == 'baseline'].copy()
print('Baseline Results:')
print('=' * 70)
print(baselines[['policy', 'task', 'success_rate', 'avg_reward', 'vram_gb', 'training_min']].to_string(index=False))

In [ ]:
# Baseline comparison bar chart
if len(baselines) > 0:
    tasks = baselines['task'].unique()
    policies = baselines['policy'].unique()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(tasks))
    width = 0.8 / len(policies)
    
    colors = ['#D97757', '#7DA488', '#5B8CB8', '#9B7EC8', '#C4956A']
    
    for i, policy in enumerate(policies):
        policy_data = baselines[baselines['policy'] == policy]
        rates = []
        for task in tasks:
            task_data = policy_data[policy_data['task'] == task]
            rates.append(task_data['success_rate'].values[0] if len(task_data) > 0 else 0)
        ax.bar(x + i * width, rates, width, label=policy, color=colors[i % len(colors)])
    
    ax.set_xlabel('Task')
    ax.set_ylabel('Success Rate (%)')
    ax.set_title('Baseline Policy Comparison')
    ax.set_xticks(x + width * (len(policies) - 1) / 2)
    ax.set_xticklabels(tasks, rotation=15, ha='right')
    ax.legend()
    ax.set_ylim(0, 105)
    plt.tight_layout()
    
    Path('reports/figures').mkdir(parents=True, exist_ok=True)
    plt.savefig('reports/figures/baseline_comparison.png', dpi=150)
    plt.show()

## 2. Optimization Progress

In [ ]:
# All kept experiments (improvements)
kept = df[df['status'] == 'keep'].copy()
print(f'Kept experiments: {len(kept)}')
if len(kept) > 0:
    print(kept[['policy', 'task', 'success_rate', 'description']].to_string(index=False))

In [ ]:
# Success rate over time (experiment index)
fig, ax = plt.subplots(figsize=(14, 6))

# Plot all experiments
colors_map = {'baseline': '#5B8CB8', 'keep': '#7DA488', 'discard': '#C4956A', 'crash': '#D97757'}

for status, color in colors_map.items():
    mask = df['status'] == status
    if mask.any():
        ax.scatter(df[mask].index, df[mask]['success_rate'], 
                   c=color, label=status, s=50, alpha=0.8)

ax.set_xlabel('Experiment Index')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Experiment Progress')
ax.legend()
plt.tight_layout()

Path('reports/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('reports/figures/experiment_progress.png', dpi=150)
plt.show()

## 3. Best Configs Found

In [ ]:
# Best result per (policy, task) pair
non_crash = df[df['status'] != 'crash'].copy()
if len(non_crash) > 0:
    best = non_crash.loc[non_crash.groupby(['policy', 'task'])['success_rate'].idxmax()]
    print('Best Results Per (Policy, Task):')
    print('=' * 80)
    print(best[['policy', 'task', 'success_rate', 'avg_reward', 'description']].to_string(index=False))

## 4. Summary Statistics

In [ ]:
print('Summary Statistics')
print('=' * 50)
print(f'Total experiments:    {len(df)}')
print(f'Baselines:            {len(df[df["status"] == "baseline"])}')
print(f'Kept (improvements):  {len(df[df["status"] == "keep"])}')
print(f'Discarded:            {len(df[df["status"] == "discard"])}')
print(f'Crashes:              {len(df[df["status"] == "crash"])}')
print(f'\nKeep rate:            {len(df[df["status"] == "keep"]) / max(1, len(df)) * 100:.1f}%')
print(f'Crash rate:           {len(df[df["status"] == "crash"]) / max(1, len(df)) * 100:.1f}%')
print(f'\nTotal training time:  {df["training_min"].sum():.0f} minutes ({df["training_min"].sum() / 60:.1f} hours)')
print(f'Peak VRAM used:       {df["vram_gb"].max():.1f} GB')

## 5. Ablation Study Plots

These plots are generated after Phase 3 ablation experiments.

In [ ]:
# Data efficiency plot (if ablation data exists)
ablation_data = df[df['description'].str.contains('data', case=False, na=False)]
if len(ablation_data) > 0:
    print('Data efficiency experiments found. Generating plot...')
    # Agent will fill in specific plotting logic based on experiment descriptions
else:
    print('No data efficiency ablation experiments yet.')